# PRAGMA · Fase A v4 — SAM 2.1 con modalidades y evidencia de alpha

**Imagen de aceptación solicitada una sola vez:** `P1070614.JPG` · 4000×2248 px  
**SHA-256:** `8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d`

**Objetivo bloqueante:** conservar por separado al señor de la izquierda y a la chica del frente, excluyendo las demás personas —incluida la persona parcialmente oculta detrás de la chica—, cuadros, mesa y demás fondo.

Esta versión valida **solo SAM 2.1 Large**. No instala YOLO-seg ni BiRefNet, no inicia FastAPI, no toca `pragma-extension.zip` y no avanza a Fase B.

### Dos preguntas separadas

1. **¿Pasa la fotografía de aceptación?** Solo si ambos sujetos son PASS.
2. **¿Puede rechazarse SAM 2 como segmentador?** Solo si un sujeto falla después de probar punto, punto con correcciones y caja.

Antes de agotar modalidades, un resultado negativo es `INCONCLUSIVE`, no `FAIL`.

### Sobre el borde suave

La v4 conserva logits de alta resolución y puede convertir su banda de transición en alpha intermedia. Esto elimina una escalera estrictamente binaria y prueba que el camino real admite alpha suave. **No es matting de pelo ni sustituye BiRefNet.** El informe registra ganancia, saturación y fracción de banda suave para que el efecto sea falsable.

Fuentes oficiales: [SAM 2 de Meta FAIR](https://github.com/facebookresearch/sam2), [SAM2ImagePredictor](https://github.com/facebookresearch/sam2/blob/main/sam2/sam2_image_predictor.py), [instalación](https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).


## 0. Uso correcto

Selecciona A100 si está disponible; L4 y T4 también sirven. La precisión continúa siendo T4→fp16, Ampere+→bf16 y CPU→fp32.

El flujo inicial por punto es interactivo. Si no sirve, prueba una o más correcciones y después una caja. La caja se selecciona mediante dos clics: esquina superior izquierda y esquina inferior derecha.

Cada intento genera una galería independiente. Nunca se elige por `argmax`. Un PASS puede lograrse con cualquier modalidad; las tres solo son obligatorias antes de declarar que SAM 2 falla.

La primera ejecución abrirá un selector para cargar `P1070614.JPG`. El nombre no importa: el cuaderno verifica el SHA-256 exacto.


In [ ]:
# Versión ligera: la foto se carga una sola vez y se valida por contenido.
EXPECTED_IMAGE_SHA256 = "8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d"


## 1. Instalar la implementación oficial

El repositorio se instala en una carpeta cuyo nombre no puede sombrear al paquete `sam2`. Después se trabaja desde `/content/pragma_run`, no desde el directorio padre del repositorio. El commit exacto se registra en el informe.


In [ ]:
import os, subprocess, sys, urllib.request
from pathlib import Path

REPO_DIR = Path("/content/pragma_sam2_official")
WORK_DIR = Path("/content/pragma_run")
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
CHECKPOINT = CHECKPOINT_DIR / "sam2.1_hiera_large.pt"
CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt"
MODEL_CFG = "configs/sam2.1/sam2.1_hiera_l.yaml"

WORK_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def run_checked(args, **kwargs):
    print("$", " ".join(map(str, args)))
    return subprocess.run(args, check=True, text=True, **kwargs)

if not (REPO_DIR / ".git").exists():
    run_checked(["git", "clone", "--depth", "1", "https://github.com/facebookresearch/sam2.git", str(REPO_DIR)])
else:
    run_checked(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])

SAM2_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

install_env = os.environ.copy()
install_env["SAM2_BUILD_CUDA"] = "0"
run_checked(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebooks]"],
    cwd=REPO_DIR,
    env=install_env,
)

# Disponible inmediatamente en este kernel; no exige reiniciar Colab.
repo_import_root = str(REPO_DIR)
if repo_import_root not in sys.path:
    sys.path.insert(0, repo_import_root)
import importlib
importlib.invalidate_caches()
import sam2
print("SAM 2 importable:", Path(sam2.__file__).resolve())

def download(url, destination):
    temporary = destination.with_suffix(destination.suffix + ".part")
    def progress(blocks, block_size, total):
        if total > 0 and blocks % 128 == 0:
            print(f"Descarga: {min(100, blocks*block_size*100/total):5.1f}%", end="\r")
    urllib.request.urlretrieve(url, temporary, reporthook=progress)
    temporary.replace(destination)
    print(f"\nCheckpoint: {destination.stat().st_size / 2**20:.1f} MiB")

if not CHECKPOINT.exists() or CHECKPOINT.stat().st_size < 800 * 2**20:
    download(CHECKPOINT_URL, CHECKPOINT)
else:
    print(f"Checkpoint reutilizado: {CHECKPOINT.stat().st_size / 2**20:.1f} MiB")

os.chdir(WORK_DIR)
print("Commit SAM 2:", SAM2_COMMIT)
print("Carpeta de ejecución:", Path.cwd())


## 2. Entorno, precisión e imagen exacta

Esta sección contiene pruebas de política: T4→fp16, A100→bf16 y CPU→fp32. La foto incorporada solo se usa si no aparece una copia con el hash esperado.


In [ ]:
EXPECTED_IMAGE_SHA256 = "8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d"
import base64, hashlib, io, json, platform, time, uuid, zipfile
from contextlib import nullcontext
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional, Protocol

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
import torch
from google.colab import files, output

def select_precision(cuda_available: bool, capability=None):
    if not cuda_available:
        return {"device": "cpu", "dtype": "float32", "autocast": False}
    if capability is None:
        raise ValueError("CUDA requiere capability=(major, minor)")
    native_bf16 = int(capability[0]) >= 8
    return {
        "device": "cuda",
        "dtype": "bfloat16" if native_bf16 else "float16",
        "autocast": True,
    }

# Demostración determinista de la corrección T4/A100/CPU.
assert select_precision(True, (7, 5))["dtype"] == "float16"
assert select_precision(True, (8, 0))["dtype"] == "bfloat16"
assert select_precision(False)["dtype"] == "float32"

CUDA_AVAILABLE = torch.cuda.is_available()
CUDA_CAPABILITY = torch.cuda.get_device_capability(0) if CUDA_AVAILABLE else None
PRECISION = select_precision(CUDA_AVAILABLE, CUDA_CAPABILITY)
DEVICE = PRECISION["device"]
TORCH_DTYPE = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}[PRECISION["dtype"]]

def inference_precision():
    if DEVICE == "cuda":
        return torch.autocast(device_type="cuda", dtype=TORCH_DTYPE)
    return nullcontext()

def synchronize():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

def gpu_peak_gib():
    return round(torch.cuda.max_memory_allocated() / 2**30, 3) if DEVICE == "cuda" else None

if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
    DEVICE_NAME = torch.cuda.get_device_name(0)
else:
    DEVICE_NAME = "CPU"
    print("ADVERTENCIA: Large en CPU puede ser muy lento; sus tiempos no son comparables con GPU.")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = WORK_DIR / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

def sha256_file(path, chunk=1024*1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()

# Buscar una copia ya cargada; si no existe, pedirla una sola vez.
image_candidates = [
    Path("/mnt/data/P1070614.JPG"),
    Path("/content/P1070614.JPG"),
    WORK_DIR / "P1070614.JPG",
]
IMAGE_PATH = next(
    (p for p in image_candidates if p.exists() and sha256_file(p) == EXPECTED_IMAGE_SHA256),
    None,
)
if IMAGE_PATH is None:
    print("Selecciona la foto P1070614.JPG. Se validará por SHA-256; el nombre puede variar.")
    uploaded = files.upload()
    exact = [(name, payload) for name, payload in uploaded.items() if hashlib.sha256(payload).hexdigest() == EXPECTED_IMAGE_SHA256]
    if len(exact) != 1:
        received = {name: hashlib.sha256(payload).hexdigest() for name, payload in uploaded.items()}
        raise ValueError(f"No se recibió exactamente la fotografía de aceptación. Hashes: {received}")
    original_name, payload = exact[0]
    IMAGE_PATH = RUN_DIR / "P1070614.JPG"
    IMAGE_PATH.write_bytes(payload)
    print(f"Foto validada: {original_name} → {IMAGE_PATH}")

IMAGE_SHA256 = sha256_file(IMAGE_PATH)
assert IMAGE_SHA256 == EXPECTED_IMAGE_SHA256, "La foto no coincide con la aceptación acordada."
image_pil = ImageOps.exif_transpose(Image.open(IMAGE_PATH)).convert("RGB")
image = np.asarray(image_pil)
assert image.shape[:2] == (2248, 4000), image.shape

ENVIRONMENT = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": DEVICE,
    "device_name": DEVICE_NAME,
    "cuda_capability": CUDA_CAPABILITY,
    "dtype": PRECISION["dtype"],
    "sam2_commit": SAM2_COMMIT,
    "checkpoint_bytes": CHECKPOINT.stat().st_size,
    "checkpoint_sha256": sha256_file(CHECKPOINT),
    "image_sha256": IMAGE_SHA256,
    "image_size": [image.shape[1], image.shape[0]],
    "run_id": RUN_ID,
}
print(json.dumps(ENVIRONMENT, indent=2, ensure_ascii=False))
plt.figure(figsize=(15, 8)); plt.imshow(image); plt.title("Imagen de aceptación verificada por SHA-256"); plt.axis("off"); plt.show()


## 3. Pipeline realmente intercambiable

Flujo ejecutado: `NoDetector → ClickSelector → SAM2MaskGenerator → IdentityRefiner → AlphaCompositor`.

El selector recibe las detecciones aunque en Fase A estén vacías; así un futuro `YoloSegDetector` puede reemplazar `NoDetector`. El refinador devuelve una matte flotante 0..1 y el compositor conserva valores intermedios, por lo que un futuro `BiRefNetMaskRefiner` no perderá bordes suaves.


In [ ]:
@dataclass
class Detection:
    xyxy: tuple
    label: str
    score: float

@dataclass
class PromptSet:
    points: np.ndarray
    labels: np.ndarray

    def __post_init__(self):
        self.points = np.asarray(self.points, dtype=np.float32).reshape(-1, 2)
        self.labels = np.asarray(self.labels, dtype=np.int32).reshape(-1)
        if len(self.points) != len(self.labels) or len(self.points) == 0:
            raise ValueError("PromptSet necesita puntos y etiquetas alineados.")
        if not set(self.labels.tolist()).issubset({0, 1}):
            raise ValueError("Las etiquetas deben ser 1 (persona) o 0 (fondo).")

@dataclass
class GeneratedMasks:
    masks: np.ndarray
    hr_logits: np.ndarray
    scores: np.ndarray
    low_res_logits: np.ndarray
    inference_s: float

@dataclass
class Proposal:
    proposal_id: str
    subject: str
    slug: str
    prompts: Optional[PromptSet]
    box: Optional[np.ndarray]
    modality: str
    masks: np.ndarray
    hr_logits: np.ndarray
    scores: np.ndarray
    logits: np.ndarray
    inference_s: float
    round_number: int
    parent_proposal_id: Optional[str] = None
    seed_candidate_index: Optional[int] = None
    gallery_path: Optional[str] = None

@dataclass
class AcceptedTrial:
    trial_id: str
    subject: str
    slug: str
    proposal_id: str
    candidate_index: int
    modality: str
    round_number: int
    points: list
    point_labels: list
    box_xyxy: Optional[list]
    candidate_scores: list
    candidate_area_fractions: list
    embedding_s: float
    prompt_inference_s: float
    compose_export_s: float
    alpha_diagnostics: dict
    acceptance_rgba_path: str
    acceptance_alpha_path: str
    feather_preview_rgba_path: str
    feather_preview_alpha_path: str
    crop_path: str
    capture_path: str
    gallery_path: str
    evidence_sha256: dict
    inspection_token: str

class Detector(Protocol):
    def detect(self, image: np.ndarray) -> list: ...

class Selector(Protocol):
    def select(self, image: np.ndarray, subject: str, detections: list) -> PromptSet: ...

class MaskGenerator(Protocol):
    def prepare(self, image: np.ndarray) -> float: ...
    def generate(self, prompts=None, mask_input=None, multimask_output=True, box=None) -> GeneratedMasks: ...

class MaskRefiner(Protocol):
    def refine(self, image: np.ndarray, matte: np.ndarray) -> np.ndarray: ...

class Compositor(Protocol):
    def compose(self, image: np.ndarray, matte: np.ndarray) -> Image.Image: ...

class NoDetector:
    def __init__(self): self.calls = 0
    def detect(self, image):
        self.calls += 1
        return []

class IdentityRefiner:
    def refine(self, image, matte):
        matte = np.asarray(matte, dtype=np.float32)
        if matte.shape != image.shape[:2]:
            raise ValueError("La matte no coincide con la imagen.")
        return np.clip(matte, 0.0, 1.0)

class AlphaCompositor:
    def compose(self, image, matte):
        matte = np.clip(np.asarray(matte, dtype=np.float32), 0.0, 1.0)
        alpha = np.rint(matte * 255.0).astype(np.uint8)
        return Image.fromarray(np.dstack([np.asarray(image, dtype=np.uint8), alpha]))

# El compositor conserva una futura matte de BiRefNet.
_alpha = np.asarray(AlphaCompositor().compose(
    np.zeros((2,2,3), np.uint8), np.full((2,2), 0.5, np.float32)
))[...,3]
assert np.all(_alpha == 128)
print("SELF-TEST PASS: matte 0.5 → alpha 128.")


### Selector recuperable de Colab

A diferencia del falso modal anterior, el contenido ocupa espacio dentro de la salida de la celda. El selector declara el texto como JSON seguro, espera a que la imagen cargue, permite cancelar y tiene timeout tanto en JavaScript como en Python.


In [ ]:
class ClickSelectionError(RuntimeError):
    pass

def build_click_js(image_data_b64: str, subject: str, timeout_ms=300_000):
    subject_json = json.dumps(subject, ensure_ascii=False)
    return f"""
    new Promise((resolve) => {{
      const subject = {subject_json};
      let finished = false;
      const root = document.createElement('section');
      root.style.cssText = 'position:relative;box-sizing:border-box;width:100%;min-height:240px;padding:14px;background:#111827;color:white;border:2px solid #10b981;border-radius:12px;font-family:Arial,sans-serif;text-align:center;';
      const title = document.createElement('div');
      title.style.cssText = 'font-size:18px;font-weight:700;margin-bottom:6px;';
      title.textContent = 'PRAGMA — Haz clic en ' + subject;
      const hint = document.createElement('div'); hint.textContent = 'Espera a que aparezca la foto.';
      const status = document.createElement('div'); status.textContent = 'Cargando imagen…'; status.style.margin='8px';
      const canvas = document.createElement('canvas');
      canvas.style.cssText='display:none;max-width:98%;height:auto;margin:auto;border:3px solid #10b981;cursor:crosshair;';
      const cancel=document.createElement('button'); cancel.textContent='Cancelar'; cancel.style.cssText='display:block;margin:12px auto 0;padding:8px 18px;';
      root.append(title,hint,status,canvas,cancel); document.body.appendChild(root);
      const finish=(result)=>{{if(finished)return;finished=true;clearTimeout(timer);canvas.style.pointerEvents='none';cancel.disabled=true;setTimeout(()=>{{root.remove();resolve(result);}},180);}};
      const timer=setTimeout(()=>finish({{status:'timeout',message:'Sin clic durante cinco minutos.'}}),{int(timeout_ms)});
      cancel.onclick=()=>finish({{status:'cancel',message:'Selección cancelada.'}});
      const img=new Image();
      img.onerror=()=>finish({{status:'image_error',message:'No se pudo decodificar la vista previa.'}});
      img.onload=()=>{{canvas.width=img.width;canvas.height=img.height;canvas.getContext('2d').drawImage(img,0,0);canvas.style.display='block';status.textContent='Esperando tu clic…';google.colab.output.setIframeHeight(document.documentElement.scrollHeight,true);}};
      img.src='data:image/jpeg;base64,{image_data_b64}';
      canvas.onclick=(event)=>{{const rect=canvas.getBoundingClientRect();const x=(event.clientX-rect.left)*canvas.width/rect.width;const y=(event.clientY-rect.top)*canvas.height/rect.height;finish({{status:'ok',x:x,y:y}});}};
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight,true);
    }})
    """

class ClickSelector:
    def __init__(self, timeout_s=90): self.timeout_s=timeout_s

    def _select_point(self, image, instruction):
        preview=Image.fromarray(image.copy()); preview.thumbnail((1000,700))
        scale_x=image.shape[1]/preview.width; scale_y=image.shape[0]/preview.height
        buffer=io.BytesIO(); preview.save(buffer,format="JPEG",quality=92)
        data=base64.b64encode(buffer.getvalue()).decode("ascii")
        print('Esperando clic. Si la imagen no aparece, detén la celda y usa la línea de respaldo indicada.')
        result=output.eval_js(
            build_click_js(data,instruction,self.timeout_s*1000),
            timeout_sec=self.timeout_s+15,
        )
        if not isinstance(result,dict) or result.get("status")!="ok":
            message=result.get("message","Selector sin respuesta") if isinstance(result,dict) else str(result)
            raise ClickSelectionError(message)
        return np.array([float(result["x"])*scale_x,float(result["y"])*scale_y],dtype=np.float32)

    def select(self,image,subject,detections=None):
        return PromptSet([self._select_point(image,f"el centro del torso de {subject}")],[1])

    def select_box(self,image,subject,detections=None):
        first=self._select_point(image,f"la esquina SUPERIOR IZQUIERDA de la caja de {subject}")
        second=self._select_point(image,f"la esquina INFERIOR DERECHA de la caja de {subject}")
        x1,y1=np.minimum(first,second); x2,y2=np.maximum(first,second)
        if x2-x1<10 or y2-y1<10: raise ClickSelectionError("La caja es demasiado pequeña.")
        return np.array([x1,y1,x2,y2],dtype=np.float32)

_js_probe=build_click_js("AA==","señor de la izquierda",1000)
assert 'const subject = "señor de la izquierda"' in _js_probe
assert '${label}' not in _js_probe and '$señor' not in _js_probe
assert "status:'timeout'" in _js_probe and "status:'cancel'" in _js_probe
print("SELF-TEST PASS: selector de punto/caja recuperable.")


## 4. Cargar SAM 2.1 Large y ejecutar smoke test con contenido

El smoke se divide en compatibilidad de API y plausibilidad sobre la foto real. Comprueba formas, finitud, máscaras no vacías, que alguna incluya el punto y que al menos una tenga área plausible. La diversidad se registra como diagnóstico, pero no elige una candidata.


In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

MASK_THRESHOLD = 0.0
ALPHA_GAIN = 3.0
ALPHA_SATURATION = 6.0

def matte_from_logits(hr_logits, gain=ALPHA_GAIN, saturation=ALPHA_SATURATION):
    z=np.asarray(hr_logits,dtype=np.float32)*float(gain)
    clipped=np.clip(z,-60.0,60.0)
    matte=1.0/(1.0+np.exp(-clipped))
    matte[z<=-float(saturation)]=0.0
    matte[z>= float(saturation)]=1.0
    return matte.astype(np.float32)

def alpha_diagnostics(matte, lo=0.02, hi=0.98):
    matte=np.asarray(matte,dtype=np.float32)
    return {
        "soft_band_fraction": float(((matte>lo)&(matte<hi)).mean()),
        "transparent_fraction": float((matte==0.0).mean()),
        "opaque_fraction": float((matte==1.0).mean()),
        "min": float(matte.min()), "max": float(matte.max()),
    }

t0=time.perf_counter()
try:
    sam2_model=build_sam2(MODEL_CFG,str(CHECKPOINT),device=DEVICE)
    predictor=SAM2ImagePredictor(sam2_model); synchronize()
except torch.cuda.OutOfMemoryError as exc:
    raise RuntimeError("FAIL_ENVIRONMENT: Large no cabe; usa A100/L4 o CPU. No se degradará a Small.") from exc
MODEL_LOAD_S=time.perf_counter()-t0

class SAM2MaskGenerator:
    def __init__(self,predictor): self.predictor=predictor
    def prepare(self,image):
        synchronize(); start=time.perf_counter()
        with torch.inference_mode(),inference_precision(): self.predictor.set_image(image)
        synchronize(); return time.perf_counter()-start
    def generate(self,prompts=None,mask_input=None,multimask_output=True,box=None):
        synchronize(); start=time.perf_counter()
        with torch.inference_mode(),inference_precision():
            hr_logits,scores,low_res=self.predictor.predict(
                point_coords=prompts.points if prompts is not None else None,
                point_labels=prompts.labels if prompts is not None else None,
                box=box,mask_input=mask_input,multimask_output=multimask_output,
                return_logits=True,
            )
        synchronize(); elapsed=time.perf_counter()-start
        raw_hr_logits=np.asarray(hr_logits,dtype=np.float32)
        binary_masks=raw_hr_logits>MASK_THRESHOLD
        stored_hr_logits=raw_hr_logits.astype(np.float16)
        return GeneratedMasks(
            masks=binary_masks,
            hr_logits=stored_hr_logits,
            scores=np.asarray(scores,dtype=np.float32),
            low_res_logits=np.asarray(low_res,dtype=np.float32),
            inference_s=elapsed,
        )

class SegmentationPipeline:
    def __init__(self,detector,selector,generator,refiner,compositor):
        self.detector=detector; self.selector=selector; self.generator=generator
        self.refiner=refiner; self.compositor=compositor; self.detections=[]; self.embedding_s=None
    def prepare(self,image):
        self.detections=self.detector.detect(image)
        self.embedding_s=self.generator.prepare(image)
        return self.embedding_s

detector=NoDetector()
pipeline=SegmentationPipeline(detector,ClickSelector(90),SAM2MaskGenerator(predictor),IdentityRefiner(),AlphaCompositor())
EMBEDDING_S=pipeline.prepare(image); assert detector.calls==1

def binary_iou(a,b):
    union=np.logical_or(a,b).sum()
    return float(np.logical_and(a,b).sum())/float(union) if union else 0.0

def evaluate_smoke(generated,prompts,image_shape):
    h,w=image_shape[:2]; areas=[float(m.mean()) for m in generated.masks]
    x,y=np.rint(prompts.points[0]).astype(int); point_inside=0<=x<w and 0<=y<h and any(bool(m[y,x]) for m in generated.masks)
    pairs=[binary_iou(generated.masks[i],generated.masks[j]) for i,j in ((0,1),(0,2),(1,2))]
    checks={
        "three_candidates": len(generated.masks)==3,
        "shapes_match": all(m.shape==(h,w) for m in generated.masks),
        "finite_scores_logits": bool(np.isfinite(generated.scores).all() and np.isfinite(generated.hr_logits).all()),
        "at_least_one_nonempty": max(areas,default=0)>1e-4,
        "point_in_at_least_one_mask": point_inside,
        "at_least_one_plausible_area": any(1e-4<a<0.60 for a in areas),
    }
    return {"status":"PASS" if all(checks.values()) else "FAIL","checks":checks,"areas":areas,"pairwise_iou":pairs}

smoke_prompts=PromptSet([[720,1050]],[1])
smoke_generated=pipeline.generator.generate(smoke_prompts,multimask_output=True)
SMOKE_PROMPT_S=smoke_generated.inference_s
SMOKE_TEST=evaluate_smoke(smoke_generated,smoke_prompts,image.shape)
SMOKE_TEST.update({"dtype":PRECISION["dtype"],"point":smoke_prompts.points.tolist()})
assert SMOKE_TEST["status"]=="PASS",SMOKE_TEST

diagnostic_index=next(i for i,m in enumerate(smoke_generated.masks) if m.mean()>1e-4)
binary_diag=alpha_diagnostics(smoke_generated.masks[diagnostic_index].astype(np.float32))
logit_diag=alpha_diagnostics(matte_from_logits(smoke_generated.hr_logits[diagnostic_index]))
assert binary_diag["soft_band_fraction"]==0.0
SMOKE_ALPHA_DIAGNOSTICS={
    "binary_v3":binary_diag,"logit_v4":logit_diag,
    "diagnostic_has_intermediate_alpha":logit_diag["soft_band_fraction"]>0.0,
    "warning":"La vista por logits no es matting y no bloquea el PASS.",
}
print(json.dumps({"smoke":SMOKE_TEST,"alpha":SMOKE_ALPHA_DIAGNOSTICS},indent=2))
del smoke_generated


## 5. Intentos, modalidades, candidatas y refinamiento

`GeneratedMasks` evita confundir el orden de cinco arrays. Cada propuesta registra modalidad, caja, puntos, logits y parentesco. Las modalidades probadas se derivan de propuestas reales con galería, no de una marca manual.


In [ ]:
from matplotlib.patches import Rectangle

REQUIRED_MODALITIES={"point","point+corrections","box"}
proposals={}; latest_proposal={}; accepted_trials={}; trial_history=[]
MODALITY_ATTEMPTS={"senor_izquierda":{},"chica_frente":{}}

def checkerboard(h,w,tile=24):
    yy,xx=np.indices((h,w)); board=((xx//tile+yy//tile)%2)[...,None]
    return np.where(board,210,245).astype(np.uint8).repeat(3,axis=2)
def matte_on_checker(image,matte):
    m=np.clip(np.asarray(matte,np.float32),0,1)[...,None]
    return np.rint(image*m+checkerboard(*image.shape[:2])*(1-m)).astype(np.uint8)
def proposal_areas(p): return [float(np.asarray(m,bool).mean()) for m in p.masks]
def prompts_json(prompts):
    return ([],[]) if prompts is None else (prompts.points.tolist(),prompts.labels.tolist())

def save_candidate_gallery(p):
    fig,axes=plt.subplots(1,len(p.masks),figsize=(6*len(p.masks),6)); axes=np.atleast_1d(axes)
    for i,(mask,score,area) in enumerate(zip(p.masks,p.scores,proposal_areas(p))):
        axes[i].imshow(matte_on_checker(image,mask.astype(np.float32)))
        axes[i].set_title(f"CANDIDATA {i}\nIoU={float(score):.3f} · área={area*100:.1f}%"); axes[i].axis("off")
    fig.suptitle(f"{p.subject} · {p.modality} · ronda {p.round_number} · ELIGE POR CONTENIDO")
    fig.tight_layout(); path=RUN_DIR/f"{p.proposal_id}_galeria.png"; fig.savefig(path,dpi=170,bbox_inches="tight")
    plt.show(); plt.close(fig); p.gallery_path=str(path); return path

def register_proposal(p):
    proposals[p.proposal_id]=p; latest_proposal[p.slug]=p.proposal_id; accepted_trials.pop(p.slug,None)
    MODALITY_ATTEMPTS.setdefault(p.slug,{}).setdefault(p.modality,[]).append(p.proposal_id)
    save_candidate_gallery(p); print("Propuesta",p.proposal_id,"· modalidad",p.modality); return p

def build_proposal(subject,slug,prompts=None,box=None,modality="point",mask_input=None,multimask_output=True,round_number=1,parent=None,seed=None):
    generated=pipeline.generator.generate(prompts,mask_input=mask_input,multimask_output=multimask_output,box=box)
    p=Proposal(
        proposal_id=f"{slug}_r{round_number:02d}_{uuid.uuid4().hex[:8]}",subject=subject,slug=slug,
        prompts=prompts,box=None if box is None else np.asarray(box,np.float32).reshape(4),modality=modality,
        masks=generated.masks,hr_logits=generated.hr_logits,scores=generated.scores,logits=generated.low_res_logits,
        inference_s=generated.inference_s,round_number=round_number,parent_proposal_id=parent,seed_candidate_index=seed,
    )
    return register_proposal(p)

def propose_with_points(subject,slug,positive_points,negative_points=None):
    negatives=list(negative_points or []); positives=list(positive_points)
    prompts=PromptSet(positives+negatives,[1]*len(positives)+[0]*len(negatives))
    modality="point" if len(positives)==1 and not negatives else "point+corrections"
    return build_proposal(subject,slug,prompts=prompts,modality=modality)
def propose_by_click(subject,slug):
    prompts=pipeline.selector.select(image,subject,pipeline.detections)
    return propose_with_points(subject,slug,prompts.points.tolist())
def propose_with_box(subject,slug,box_xyxy,positive_points=None,negative_points=None):
    positives=list(positive_points or []); negatives=list(negative_points or [])
    prompts=PromptSet(positives+negatives,[1]*len(positives)+[0]*len(negatives)) if positives or negatives else None
    modality="box" if prompts is None else "box+points"
    return build_proposal(subject,slug,prompts=prompts,box=np.asarray(box_xyxy,np.float32),modality=modality)
def propose_by_box_click(subject,slug):
    box=pipeline.selector.select_box(image,subject,pipeline.detections)
    return propose_with_box(subject,slug,box)

def refine_with_click(p,seed_candidate_index,point_label):
    if seed_candidate_index not in range(len(p.masks)): raise IndexError("Candidata semilla fuera de rango.")
    if point_label not in (0,1): raise ValueError("1=persona; 0=fondo.")
    instruction=f"la parte que falta de {p.subject} (+)" if point_label else "el objeto/persona que debe quitarse (−)"
    extra=pipeline.selector.select(image,instruction,pipeline.detections)
    base_points=np.empty((0,2),np.float32) if p.prompts is None else p.prompts.points
    base_labels=np.empty((0,),np.int32) if p.prompts is None else p.prompts.labels
    prompts=PromptSet(np.vstack([base_points,extra.points]),np.concatenate([base_labels,[point_label]]))
    modality="box+points" if p.box is not None else "point+corrections"
    return build_proposal(
        p.subject,p.slug,prompts=prompts,box=p.box,modality=modality,
        mask_input=p.logits[seed_candidate_index][None,:,:],multimask_output=False,
        round_number=p.round_number+1,parent=p.proposal_id,seed=seed_candidate_index,
    )

def bbox_from_matte(matte,padding=20,threshold=0.02):
    ys,xs=np.where(matte>threshold)
    if not len(xs): raise ValueError("Matte vacía.")
    return max(0,int(xs.min())-padding),max(0,int(ys.min())-padding),min(matte.shape[1],int(xs.max())+padding+1),min(matte.shape[0],int(ys.max())+padding+1)

def finalize_candidate(p,candidate_index):
    if p.proposal_id!=latest_proposal.get(p.slug): raise RuntimeError("Propuesta caduca; inspecciona la última ronda.")
    if candidate_index is None or candidate_index not in range(len(p.masks)): raise ValueError(f"Elige 0..{len(p.masks)-1}.")
    start=time.perf_counter(); binary_matte=pipeline.refiner.refine(image,p.masks[candidate_index].astype(np.float32))
    feather_matte=matte_from_logits(p.hr_logits[candidate_index])
    alpha_stats={"acceptance_binary":alpha_diagnostics(binary_matte),"logit_feather_preview":alpha_diagnostics(feather_matte)}
    # La salida bloqueante sigue siendo binaria. La versión suavizada es evidencia diagnóstica, no matting ni gate PASS.
    trial_id=f"trial_{uuid.uuid4().hex[:10]}"; stem=f"{p.proposal_id}_c{candidate_index}_{trial_id}"
    acceptance_rgba=pipeline.compositor.compose(image,binary_matte); feather_rgba=pipeline.compositor.compose(image,feather_matte)
    acceptance_rgba_path=RUN_DIR/f"{stem}_acceptance_binary_rgba.png"; acceptance_alpha_path=RUN_DIR/f"{stem}_acceptance_binary_alpha.png"
    feather_rgba_path=RUN_DIR/f"{stem}_logit_feather_preview_rgba.png"; feather_alpha_path=RUN_DIR/f"{stem}_logit_feather_preview_alpha.png"
    crop_path=RUN_DIR/f"{stem}_crop.png"; capture_path=RUN_DIR/f"{stem}_captura.png"
    acceptance_rgba.save(acceptance_rgba_path); Image.fromarray(np.rint(binary_matte*255).astype(np.uint8)).save(acceptance_alpha_path)
    feather_rgba.save(feather_rgba_path); Image.fromarray(np.rint(feather_matte*255).astype(np.uint8)).save(feather_alpha_path)
    acceptance_rgba.crop(bbox_from_matte(binary_matte,threshold=0.5)).save(crop_path)
    overlay=image.copy(); binary=p.masks[candidate_index]; overlay[binary]=np.rint(.55*overlay[binary]+.45*np.array([0,255,90])).astype(np.uint8)
    fig,ax=plt.subplots(1,5,figsize=(27,6)); ax[0].imshow(image); ax[0].set_title("Prompts/caja · coordenadas")
    if p.prompts is not None: ax[0].scatter(p.prompts.points[:,0],p.prompts.points[:,1],c=["lime" if v else "red" for v in p.prompts.labels],s=90,marker="*")
    if p.box is not None: ax[0].add_patch(Rectangle((p.box[0],p.box[1]),p.box[2]-p.box[0],p.box[3]-p.box[1],fill=False,edgecolor="cyan",linewidth=3))
    ax[1].imshow(overlay); ax[1].set_title("Máscara binaria"); ax[1].axis("off")
    ax[2].imshow(matte_on_checker(image,binary_matte)); ax[2].set_title("ACEPTACIÓN · alpha binaria"); ax[2].axis("off")
    ax[3].imshow(matte_on_checker(image,feather_matte)); ax[3].set_title("DIAGNÓSTICO · feather por logits"); ax[3].axis("off")
    ax[4].imshow(feather_matte,cmap="gray",vmin=0,vmax=1); ax[4].set_title(f"Banda intermedia {alpha_stats['logit_feather_preview']['soft_band_fraction']:.4%}"); ax[4].axis("off")
    fig.suptitle(f"{p.subject} · {p.modality} · ronda {p.round_number} · candidata {candidate_index}"); fig.tight_layout(); fig.savefig(capture_path,dpi=170,bbox_inches="tight"); plt.show(); plt.close(fig)
    reopened_binary=Image.open(acceptance_rgba_path); binary_alpha=np.asarray(reopened_binary)[...,3]
    reopened_feather=Image.open(feather_rgba_path); feather_alpha=np.asarray(reopened_feather)[...,3]
    assert reopened_binary.mode=="RGBA" and reopened_binary.size==image_pil.size and binary_alpha.min()==0 and binary_alpha.max()==255
    assert reopened_feather.mode=="RGBA" and reopened_feather.size==image_pil.size
    alpha_stats["logit_feather_preview"]["intermediate_fraction_u8"]=float(((feather_alpha>0)&(feather_alpha<255)).mean())
    compose_s=time.perf_counter()-start
    evidence=[acceptance_rgba_path,acceptance_alpha_path,feather_rgba_path,feather_alpha_path,crop_path,capture_path,Path(p.gallery_path)]
    pts,lbl=prompts_json(p.prompts); evidence_hashes={q.name:sha256_file(q) for q in evidence}
    token_payload=json.dumps({"proposal_id":p.proposal_id,"candidate_index":int(candidate_index),"evidence_sha256":evidence_hashes},sort_keys=True).encode()
    inspection_token=hashlib.sha256(token_payload).hexdigest()
    trial=AcceptedTrial(
        trial_id=trial_id,subject=p.subject,slug=p.slug,proposal_id=p.proposal_id,candidate_index=int(candidate_index),modality=p.modality,round_number=p.round_number,
        points=pts,point_labels=lbl,box_xyxy=None if p.box is None else p.box.tolist(),candidate_scores=[float(v) for v in p.scores],
        candidate_area_fractions=proposal_areas(p),embedding_s=EMBEDDING_S,prompt_inference_s=p.inference_s,compose_export_s=compose_s,
        alpha_diagnostics=alpha_stats,acceptance_rgba_path=str(acceptance_rgba_path),acceptance_alpha_path=str(acceptance_alpha_path),
        feather_preview_rgba_path=str(feather_rgba_path),feather_preview_alpha_path=str(feather_alpha_path),
        crop_path=str(crop_path),capture_path=str(capture_path),gallery_path=p.gallery_path,evidence_sha256=evidence_hashes,inspection_token=inspection_token,
    )
    trial_history.append(trial); accepted_trials[p.slug]=trial
    print("Candidata aceptada provisionalmente. Copia este inspection_token al checklist:",inspection_token)
    return trial


## 6A. Señor de la izquierda

Empieza con punto. Si no pasa, ejecuta correcciones y una caja antes de declarar FAIL. Tras cualquier nueva propuesta, vuelve a elegir el índice y finaliza la última ronda.


In [ ]:
# Preferido: selección visible mediante clic.
proposal_senor = propose_by_click("señor de la izquierda", "senor_izquierda")
# RESPALDO BRAVE (solo si el panel no aparece): detén la celda, comenta la línea anterior y descomenta esta.
# proposal_senor = propose_with_points("señor de la izquierda", "senor_izquierda", [[720, 1050]])


In [ ]:
SENOR_MASK_INDEX=None  # 0, 1 o 2; una ronda refinada solo tiene 0
if SENOR_MASK_INDEX is None: print("PENDIENTE: elige explícitamente una candidata.")
else: trial_senor=finalize_candidate(proposal_senor,SENOR_MASK_INDEX)


In [ ]:
# OPCIONES SI EL PUNTO NO SIRVE. Ejecuta y después vuelve a la celda anterior.
# proposal_senor = refine_with_click(proposal_senor, seed_candidate_index=0, point_label=0)  # quitar fondo/persona
# proposal_senor = refine_with_click(proposal_senor, seed_candidate_index=0, point_label=1)  # añadir parte faltante
# proposal_senor = propose_by_box_click("señor de la izquierda", "senor_izquierda")


## 6B. Chica del frente

Revisa pelo y hombro contra la tercera persona, mano sobreexpuesta y cuadros. Si falla el punto, prueba correcciones y caja antes de emitir FAIL.


In [ ]:
# Preferido: selección visible mediante clic.
proposal_chica = propose_by_click("chica del frente", "chica_frente")
# RESPALDO BRAVE (solo si el panel no aparece): detén la celda, comenta la línea anterior y descomenta esta.
# proposal_chica = propose_with_points("chica del frente", "chica_frente", [[2750, 1500]])


In [ ]:
CHICA_MASK_INDEX=None  # 0, 1 o 2; una ronda refinada solo tiene 0
if CHICA_MASK_INDEX is None: print("PENDIENTE: elige explícitamente una candidata.")
else: trial_chica=finalize_candidate(proposal_chica,CHICA_MASK_INDEX)


In [ ]:
# OPCIONES SI EL PUNTO NO SIRVE. Ejecuta y después vuelve a la celda anterior.
# proposal_chica = refine_with_click(proposal_chica, seed_candidate_index=0, point_label=0)
# proposal_chica = refine_with_click(proposal_chica, seed_candidate_index=0, point_label=1)
# proposal_chica = propose_by_box_click("chica del frente", "chica_frente")


## 7. Checklist, `INCONCLUSIVE` y veredicto

Un PASS no necesita agotar modalidades, pero el checklist debe copiar `proposal_id`, `candidate_index` e `inspection_token` impresos al finalizar. Así una evidencia nueva no puede heredar un PASS antiguo.

Para declarar FAIL, `MODALITY_REVIEWS` debe ligar cada modalidad a una propuesta real, enumerar **todas** sus candidatas revisadas y contener notas. Un simple conjunto de nombres no basta. Mientras falte una revisión válida, el estado es `INCONCLUSIVE`.


In [ ]:
CHECKLIST={
    "senor_izquierda":{"proposal_id":None,"candidate_index":None,"inspection_token":None,"correct_subject":None,"body_and_edges_complete":None,"other_people_excluded":None,"background_and_frames_excluded":None,"transparent_png_and_evidence":None,"notes":""},
    "chica_frente":{"proposal_id":None,"candidate_index":None,"inspection_token":None,"correct_subject":None,"body_and_edges_complete":None,"other_people_excluded":None,"background_and_frames_excluded":None,"transparent_png_and_evidence":None,"notes":""},
}
CRITERIA=["correct_subject","body_and_edges_complete","other_people_excluded","background_and_frames_excluded","transparent_png_and_evidence"]

# Para declarar FAIL, cada modalidad necesita una revisión ligada a una propuesta real
# y a todos sus índices de candidata. Un nombre dentro de un set no basta.
MODALITY_REVIEWS={
    slug:{mode:{"proposal_id":None,"result":None,"candidate_indices_reviewed":[],"notes":""} for mode in sorted(REQUIRED_MODALITIES)}
    for slug in CHECKLIST
}

def modality_failure_audit(slug,reviews,attempts,proposal_store):
    missing=[]; verified={}
    for mode in sorted(REQUIRED_MODALITIES):
        review=reviews[slug][mode]; proposal_id=review["proposal_id"]
        reason=None
        if review["result"]!="FAIL": reason="result debe ser FAIL"
        elif proposal_id not in attempts.get(slug,{}).get(mode,[]): reason="proposal_id no pertenece a esta modalidad"
        elif proposal_id not in proposal_store: reason="proposal_id desconocido"
        elif sorted(set(review["candidate_indices_reviewed"]))!=list(range(len(proposal_store[proposal_id].masks))): reason="faltan candidatas por revisar"
        elif not review["notes"].strip(): reason="notes obligatorio"
        if reason: missing.append({"modality":mode,"reason":reason})
        else: verified[mode]=proposal_id
    return {"complete":not missing,"verified_failures":verified,"missing_or_invalid":missing}

def subject_evaluation(slug,checklist,trials,modality_attempts,reviews,proposal_store,latest):
    values=[checklist[slug][k] for k in CRITERIA]
    invalid=[v for v in values if v not in (True,False,None)]
    if invalid: raise ValueError(f"Valores inválidos: {invalid}")
    tried=set(modality_attempts.get(slug,{})); missing=sorted(REQUIRED_MODALITIES-tried)
    if all(v is True for v in values):
        if slug not in trials: raise ValueError(f"{slug}: PASS sin candidata/evidencias.")
        trial=trials[slug]
        binding=(checklist[slug]["proposal_id"]==trial.proposal_id and checklist[slug]["candidate_index"]==trial.candidate_index and checklist[slug]["inspection_token"]==trial.inspection_token)
        if not binding: raise ValueError(f"{slug}: checklist obsoleto; proposal/candidate/token no coinciden con la evidencia aceptada.")
        return {"status":"PASS","modalities_tried":sorted(tried),"modalities_missing":missing,"bound_trial_id":trial.trial_id}
    if any(v is False for v in values):
        if not checklist[slug]["notes"].strip(): raise ValueError(f"{slug}: notes obligatorio para resultado negativo.")
        if checklist[slug]["proposal_id"]!=latest.get(slug): raise ValueError(f"{slug}: el checklist negativo debe referirse a la propuesta más reciente.")
        audit=modality_failure_audit(slug,reviews,modality_attempts,proposal_store)
        return {"status":"FAIL" if audit["complete"] else "INCONCLUSIVE","modalities_tried":sorted(tried),"modalities_missing":missing,"failure_audit":audit}
    return {"status":"PENDING","modalities_tried":sorted(tried),"modalities_missing":missing,"reason":"checklist incompleto"}

_ORDER=["FAIL","INCONCLUSIVE","PENDING","PASS"]
def overall_status(statuses): return min(statuses.values(),key=_ORDER.index)
assert overall_status({"a":"FAIL","b":"PENDING"})=="FAIL"
assert overall_status({"a":"INCONCLUSIVE","b":"PASS"})=="INCONCLUSIVE"
assert overall_status({"a":"PASS","b":"PENDING"})=="PENDING"
assert overall_status({"a":"PASS","b":"PASS"})=="PASS"

SUBJECT_EVALUATION={slug:subject_evaluation(slug,CHECKLIST,accepted_trials,MODALITY_ATTEMPTS,MODALITY_REVIEWS,proposals,latest_proposal) for slug in CHECKLIST}
SUBJECT_STATUS={slug:value["status"] for slug,value in SUBJECT_EVALUATION.items()}
OVERALL_STATUS=overall_status(SUBJECT_STATUS)
print(json.dumps({"subjects":SUBJECT_EVALUATION,"overall":OVERALL_STATUS,"phase_b_blocked":OVERALL_STATUS!="PASS","sam2_rejectable":OVERALL_STATUS=="FAIL"},indent=2,ensure_ascii=False))


## 8. Informe, manifiesto y ZIP sin residuos

El reporte separa `phase_b_blocked` de `sam2_rejectable`, registra modalidades e incluye estadísticas de alpha. El ZIP conserva la lista blanca y los hashes de la v3.


In [ ]:
from dataclasses import asdict
SUBJECT_EVALUATION={slug:subject_evaluation(slug,CHECKLIST,accepted_trials,MODALITY_ATTEMPTS,MODALITY_REVIEWS,proposals,latest_proposal) for slug in CHECKLIST}
SUBJECT_STATUS={slug:value["status"] for slug,value in SUBJECT_EVALUATION.items()}
OVERALL_STATUS=overall_status(SUBJECT_STATUS)
REPORT={
    "phase":"PRAGMA Fase A v4 / SAM 2.1","overall":OVERALL_STATUS,"subject_status":SUBJECT_STATUS,"subject_evaluation":SUBJECT_EVALUATION,
    "phase_b_blocked":OVERALL_STATUS!="PASS","sam2_rejectable":OVERALL_STATUS=="FAIL",
    "required_modalities_before_fail":sorted(REQUIRED_MODALITIES),
    "modalities_tried":{slug:{mode:list(ids) for mode,ids in modes.items()} for slug,modes in MODALITY_ATTEMPTS.items()},"modality_reviews":MODALITY_REVIEWS,
    "model":{"name":"sam2.1_hiera_large","parameters_millions":224.4,"config":MODEL_CFG,"checkpoint_url":CHECKPOINT_URL,"cuda_extension_built":False},
    "alpha":{"acceptance_output":"binary SAM 2 mask","diagnostic_preview":"sigmoid of high-resolution logits; not hair matting and not a PASS gate","gain":ALPHA_GAIN,"saturation":ALPHA_SATURATION,"mask_threshold":MASK_THRESHOLD},
    "environment":ENVIRONMENT,"timings":{"model_load_s":MODEL_LOAD_S,"image_embedding_shared_s":EMBEDDING_S,"smoke_prompt_s":SMOKE_PROMPT_S,"gpu_peak_gib":gpu_peak_gib()},
    "smoke_test":SMOKE_TEST,"smoke_alpha_diagnostics":SMOKE_ALPHA_DIAGNOSTICS,"checklist":CHECKLIST,
    "accepted_trials":{slug:asdict(trial) for slug,trial in accepted_trials.items()},"trial_history":[asdict(trial) for trial in trial_history],
    "proposal_history":[{
        "proposal_id":p.proposal_id,"subject":p.subject,"slug":p.slug,"modality":p.modality,"round":p.round_number,"parent":p.parent_proposal_id,
        "seed_candidate_index":p.seed_candidate_index,"box":None if p.box is None else p.box.tolist(),
        "points":prompts_json(p.prompts)[0],"labels":prompts_json(p.prompts)[1],"scores":[float(v) for v in p.scores],"areas":proposal_areas(p),
        "inference_s":p.inference_s,"gallery":p.gallery_path,
    } for p in proposals.values()],
}
report_path=RUN_DIR/"reporte_fase_a_v4.json"; report_path.write_text(json.dumps(REPORT,indent=2,ensure_ascii=False),encoding="utf-8")
whitelist={report_path}
for p in proposals.values():
    if p.gallery_path: whitelist.add(Path(p.gallery_path))
def trial_paths(t):
    return list(map(Path,[t.acceptance_rgba_path,t.acceptance_alpha_path,t.feather_preview_rgba_path,t.feather_preview_alpha_path,t.crop_path,t.capture_path,t.gallery_path]))
# Ledger append-only: todas las finalizaciones permanecen auditables.
for t in trial_history:
    for p in trial_paths(t):
        expected_hash=t.evidence_sha256.get(p.name)
        if expected_hash is None or sha256_file(p)!=expected_hash: raise RuntimeError(f"Evidencia alterada después de finalizar: {p.name}")
        whitelist.add(p)
entries=[{"file":p.name,"bytes":p.stat().st_size,"sha256":sha256_file(p)} for p in sorted(whitelist,key=lambda p:p.name)]
manifest={"run_id":RUN_ID,"overall":OVERALL_STATUS,"files":entries,"exact_file_count_excluding_manifest":len(entries)}
manifest_path=RUN_DIR/"manifest.json"; manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding="utf-8")
zip_path=WORK_DIR/f"PRAGMA_Fase_A_v4_{RUN_ID}_{OVERALL_STATUS}.zip"
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(manifest_path,manifest_path.name)
    for p in sorted(whitelist,key=lambda p:p.name): archive.write(p,p.name)
with zipfile.ZipFile(zip_path) as archive:
    expected={"manifest.json"}|{p.name for p in whitelist}; assert set(archive.namelist())==expected and len(archive.namelist())==len(expected)
    entry_by_name={item["file"]:item for item in entries}
    for name,item in entry_by_name.items():
        payload=archive.read(name)
        assert len(payload)==item["bytes"] and hashlib.sha256(payload).hexdigest()==item["sha256"]
print(json.dumps({"overall":OVERALL_STATUS,"phase_b_blocked":OVERALL_STATUS!="PASS","sam2_rejectable":OVERALL_STATUS=="FAIL","zip":str(zip_path),"zip_sha256":sha256_file(zip_path)},indent=2))
files.download(str(zip_path))


## Interpretación honesta

- El smoke puede fallar por contenido; ya no basta contar tres tensores.
- La salida de aceptación conserva la máscara binaria de SAM 2. La alpha derivada de logits se exporta aparte como diagnóstico; no es matting de pelo ni requisito PASS.
- Un PASS puede lograrse con punto, correcciones o caja.
- Un resultado negativo antes de probar las tres modalidades es `INCONCLUSIVE`.
- `sam2_rejectable=true` solo aparece cuando las tres modalidades tienen revisiones FAIL ligadas a propuestas y candidatas concretas.
- Un checklist PASS está ligado a `proposal_id`, `candidate_index` y hash de evidencias mediante `inspection_token`; no puede heredarse tras un reintento.
- La foto sigue siendo el único caso bloqueante. Otras imágenes pueden añadirse después como regresiones no bloqueantes, sin alterar este contrato.
- YOLO-seg, BiRefNet y Fase B siguen fuera de este cuaderno.
